# **Library install**

In [1]:
!pip install sentence-transformers
!pip install transformers
!pip install scikit-learn
!pip install pandas
!pip install numpy

# **Import Library**

In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# **Load Dataset**

In [ ]:
amazon = pd.read_csv(
    "/content/data/amazon_cells_labelled.txt",
    names=["text", "label"],
    sep="\t"
)

imdb = pd.read_csv(
    "/content/data/imdb_labelled.txt",
    names=["text", "label"],
    sep="\t"
)

yelp = pd.read_csv(
    "/content/data/yelp_labelled.txt",
    names=["text", "label"],
    sep="\t"
)

print("Amazon:", amazon.shape)
print("IMDB:", imdb.shape)
print("Yelp:", yelp.shape)

amazon.columns = ["text", "label"]
imdb.columns = ["text", "label"]
yelp.columns = ["text", "label"]

data = pd.concat([amazon, imdb, yelp])

data.head()

,text,label
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


In [4]:
print("Jumlah data:", len(data))

data["label"].value_counts()

Jumlah data: 2748


,count
label,
1,1386
0,1362


# **Split Data Training dan Testing**

In [5]:
texts = data["text"]
labels = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)
print("Jumlah data train:", len(X_train))
print("Jumlah data test:", len(X_test))

Jumlah data train: 2198
Jumlah data test: 550


# **Load Hugging Face Model**

In [6]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

print("Model berhasil dimuat:", model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model berhasil dimuat: sentence-transformers/all-MiniLM-L6-v2


# **Membuat Embedding**

In [7]:
X_train_embeddings = model.encode(X_train.tolist())
X_test_embeddings = model.encode(X_test.tolist())

print("Shape embedding:", X_train_embeddings.shape)

Shape embedding: (2198, 384)


# **Training Model**

In [8]:
classifier = LogisticRegression(max_iter=1000)

classifier.fit(X_train_embeddings, y_train)

print("Model berhasil dilatih")

Model berhasil dilatih


# **Prediction**

In [9]:
predictions = classifier.predict(X_test_embeddings)

print("Contoh hasil prediksi:")
print(predictions[:10])

Contoh hasil prediksi:
[1 1 0 0 0 0 0 1 1 0]


# **Evaluasi Model**

In [10]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, predictions))

print("\nClassification Report:\n")
print(classification_report(y_test, predictions))

Accuracy: 0.9218181818181819

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.91      0.92       291
           1       0.90      0.94      0.92       259

    accuracy                           0.92       550
   macro avg       0.92      0.92      0.92       550
weighted avg       0.92      0.92      0.92       550



# **Test**

In [11]:
example_text = [
    "This phone is fantastic and very useful"
]

example_embedding = model.encode(example_text)

prediction = classifier.predict(example_embedding)

print("Text:", example_text[0])

if prediction[0] == 1:
    print("Predicted Sentiment: Positive")
else:
    print("Predicted Sentiment: Negative")

Text: This phone is fantastic and very useful
Predicted Sentiment: Positive


In [12]:
examples = [
    "This movie is amazing and very entertaining",
    "This movie is boring and a waste of time"
]

embeddings = model.encode(examples)

predictions = classifier.predict(embeddings)

for text, pred in zip(examples, predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print("Text:", text)
    print("Predicted Sentiment:", sentiment)
    print()

Text: This movie is amazing and very entertaining
Predicted Sentiment: Positive

Text: This movie is boring and a waste of time
Predicted Sentiment: Negative



# **Simpan Dataset Sample**

In [13]:
sample = data.sample(20)

sample.to_csv("dataset_sample.csv", index=False)

print("Dataset sample berhasil disimpan")

Dataset sample berhasil disimpan
